In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import cellrank as cr
from cellrank.kernels import RealTimeKernel, ConnectivityKernel
from cellrank.estimators import GPCCA
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.sparse import issparse
import matplotlib as mpl

In [ ]:
from pyfuncs.general import preprocessing_adata_sub, plot_volcano


In [ ]:
from matplotlib import font_manager, rcParams

sns.set_style("white")

font_path = "src/fonts/texgyreheros-regular.otf" # Alternative
font_path = "/usr/share/texmf/fonts/opentype/public/tex-gyre/texgyreheros-regular.otf"

mpl.rcParams.update({'font.size': 22})


# Path to the Helvetica font file
custom_font = font_manager.FontProperties(fname=font_path)

# Add the font to Matplotlib's font manager
font_manager.fontManager.addfont(font_path)

# Set the font globally
rcParams['font.family'] = custom_font.get_name()
rcParams['font.size'] = 16
rcParams['figure.dpi'] = 300

## Compute FLE coordinates

In [ ]:
adata_var = sc.read('data/oprescu-WOT/adata_oprescu_onlyFAP_clean.h5ad')
adata_var

In [ ]:
adata_var.obs['batch'].cat.categories

In [ ]:
dict_batch_to_day = {'Noninjured': 100,
                     'X0.5.DPI': 0.5, 
                     'X2.DPI': 2, 
                     'X3.5.DPI': 3.5, 
                     'X5.DPI': 5, 
                     'X10.DPI': 10,
                     'X21.DPI': 21}

adata_var.obs['day_num'] = [dict_batch_to_day[i] for i in adata_var.obs['batch']]
adata_var.obs['day'] = adata_var.obs['day_num'].astype(str)

In [ ]:
adata_var

In [ ]:
INTEGRATE = False

if INTEGRATE:
    rep_pca = 'X_pca_harmony'
else:
    rep_pca = 'X_pca'



In [ ]:
adata_var_0 = adata_var[adata_var.obs['day_num'] == 100]
adata_var_0.obs['day_num'] = 0
adata_var_0.obs['day'] = '0'

adata_var_sup = sc.concat([adata_var_0, adata_var])

In [ ]:
preprocessing_adata_sub(adata_var_sup, integrate=INTEGRATE)  # Try with integration


In [ ]:
sc.tl.umap(adata_var_sup)
sc.pl.umap(adata_var_sup, color='merged_clusters')
sc.pl.umap(adata_var_sup, color='day')

In [ ]:
adata_var_sup.obs_names_make_unique()

In [ ]:
from moscot.problems.time import TemporalProblem

# ── 1. Construir el TemporalProblem ────────────────────────

tp = TemporalProblem(adata_var_sup)

tp = tp.prepare(
    time_key  = "day_num",
    joint_attr = {"attr": "obsm", "key": rep_pca}, 
)

# ── 2. Resolver (transporte óptimo entre timepoints contiguos) ─

tp = tp.solve(
    epsilon   = 0.1,      # regularización entropica; aumentar si es lento
    tau_a     = 0.99,     # masa que puede "desaparecer" en origen
    tau_b     = 0.99,     # masa que puede "aparecer" en destino
    scale_cost = "mean",  # normaliza la matriz de costes
)

print(tp)           # debe mostrar los pares resueltos: (0.0, 0.5), (0.5, 2.0), etc.
print(tp.solutions) # dict con los transport maps


In [ ]:
rtk = RealTimeKernel.from_moscot(tp) 

rtk.compute_transition_matrix(
    threshold        = 1e-6,
    self_transitions = 'all',
    conn_weight      = 0.2,
)

print(rtk)

In [ ]:
ck = ConnectivityKernel(adata_var_sup).compute_transition_matrix()
ck

In [ ]:
# We maintain the ConnectivityKernel if someone wants to play
# but there are almost no differences between the two.
# The results look slightly better with rtk
# Also, interestingly, integrating or not integrating does not affect the results.

combined_kernel = rtk     # X * rtk + Y * ck  
combined_kernel.compute_transition_matrix()  # normaliza la combinación

print(combined_kernel)


In [ ]:
# ── La matriz de transición global ────────────────────────
T = combined_kernel.transition_matrix  # sparse (n_cells × n_cells)

days_sorted = sorted(adata_var_sup.obs["day_num"].unique())
pairs = list(zip(days_sorted[:-1], days_sorted[1:]))  # [(0,0.5), (0.5,2), ...]

transition_blocks = {}

for d_src, d_tgt in pairs:

    idx_src = np.where(adata_var_sup.obs["day_num"] == d_src)[0]
    idx_tgt = np.where(adata_var_sup.obs["day_num"] == d_tgt)[0]

    # Submatriz: filas = células origen, columnas = células destino
    block = T[np.ix_(idx_src, idx_tgt)]
    if issparse(block):
        block = block.toarray()

    # Renormalizar filas a 1 (la masa que no va a días no contiguos se redistribuye)
    row_sums = block.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # evitar división por cero
    block = block / row_sums

    transition_blocks[(d_src, d_tgt)] = {
        "matrix"   : block,
        "obs_src"  : adata_var_sup.obs_names[idx_src].tolist(),
        "obs_tgt"  : adata_var_sup.obs_names[idx_tgt].tolist(),
    }

    print(f"d{d_src} → d{d_tgt}: {block.shape}  "
          f"(min={block.min():.4f}, max={block.max():.4f})")

In [ ]:
pop_key = "merged_clusters"  

pop_transitions = {}

for (d_src, d_tgt), data in transition_blocks.items():

    pops_src = adata_var_sup.obs.loc[data["obs_src"], pop_key].values
    pops_tgt = adata_var_sup.obs.loc[data["obs_tgt"], pop_key].values

    unique_src = sorted(set(pops_src))
    unique_tgt = sorted(set(pops_tgt))

    pop_mat = pd.DataFrame(0.0, index=unique_src, columns=unique_tgt)

    # One-hot encoding con orden fijo
    dummies = pd.get_dummies(pops_tgt).reindex(columns=unique_tgt, fill_value=0).values
    # dummies: (n_cells_tgt, n_pops_tgt)

    for pop_s in unique_src:
        mask_src = pops_src == pop_s
        if mask_src.sum() == 0:
            continue

        # (n_src_cells_in_pop,) @ (n_tgt_cells, n_pops_tgt) promediando primero
        mean_probs = data["matrix"][mask_src, :].mean(axis=0)  # (n_tgt_cells,)
        pop_row    = mean_probs @ dummies                       # (n_pops_tgt,)

        pop_mat.loc[pop_s, :] = pop_row

    # Renormalizar filas
    pop_mat = pop_mat.div(pop_mat.sum(axis=1), axis=0).fillna(0)

    pop_transitions[(d_src, d_tgt)] = pop_mat
    print(f"\nd{d_src} → d{d_tgt}:")
    print(pop_mat.round(3))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, (d_src, d_tgt) in zip(axes.flatten(), pairs):
    mat = pop_transitions[(d_src, d_tgt)]
    sns.heatmap(
        mat,
        ax       = ax,
        cmap     = "YlOrRd",
        vmin     = 0, vmax = 1,
        annot    = True,
        fmt      = ".2f",
        linewidths = 0.5,
    )
    ax.set_title(f"d{d_src} → d{d_tgt}")
    ax.set_xlabel("Origin population")
    ax.set_ylabel("End population")

axes.flatten()[-1].set_axis_off()


plt.tight_layout()
# plt.savefig("transition_matrices_by_day.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# ── Parámetros ─────────────────────────────────────────────────────────

pop_key      = "merged_clusters"
time_key     = "day_num"
threshold    = 0.2   # descartar transiciones muy débiles
x_spacing    = 2.5
y_spacing    = 2.5

# Orden manual de columnas y filas
cluster_order = [ 'SX', 'S0', 'S1',  'S2', 'Krano', 'FAP_A', 'FAP_B', 'FAP_CDEF', ]

days_sorted = sorted(adata_var_sup.obs[time_key].unique())

# ── Posiciones fijas ───────────────────────────────────────────────────

# Filtrar cluster_order a los que realmente existen
existing = adata_var.obs[pop_key].unique().tolist()
cluster_order = [c for c in cluster_order if c in existing]
# Añadir los que falten al final
cluster_order += [c for c in existing if c not in cluster_order]

cluster_x = {cl: i * x_spacing for i, cl in enumerate(cluster_order)}
day_y     = {d:  -i * y_spacing for i, d in enumerate(days_sorted)}

# ── Contar células por (día, población) para tamaño de nodo ───────────

cell_counts = (
    adata_var_sup.obs.groupby([time_key, pop_key])
    .size()
    .reset_index(name="n_cells")
)
count_dict = {(row[time_key], row[pop_key]): row["n_cells"]
              for _, row in cell_counts.iterrows()}

# ── Construir grafo ────────────────────────────────────────────────────

G   = nx.Graph()
pos = {}

# Nodos: solo los (día, pop) que existen
for day in days_sorted:
    pops_present = adata_var_sup.obs.loc[adata_var_sup.obs[time_key] == day, pop_key].unique()
    for pop in pops_present:
        node = (day, pop)
        G.add_node(node, day=day, pop=pop)
        pos[node] = (cluster_x[pop], day_y[day])

# Aristas: de pop_transitions ya calculado
for (d_src, d_tgt), mat in pop_transitions.items():
    for pop_src in mat.index:
        for pop_tgt in mat.columns:
            w = mat.loc[pop_src, pop_tgt]
            if w <= threshold:
                continue
            n_src = (d_src, pop_src)
            n_tgt = (d_tgt, pop_tgt)
            if n_src in G and n_tgt in G:
                G.add_edge(n_src, n_tgt, weight=float(w))

# ── Plot ───────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(16, 10))

# Colores por día
cmap      = plt.get_cmap("tab10")
day_color = {d: cmap(i % 10) for i, d in enumerate(days_sorted)}

# Edges: grosor y gris proporcional al peso
weights = [d["weight"] for _, _, d in G.edges(data=True)]
w_min, w_max = (min(weights), max(weights)) if weights else (0, 1)

for u, v, d in G.edges(data=True):
    w      = d["weight"]
    w_norm = (w - w_min) / (w_max - w_min + 1e-9)
    gray   = 0.8 - 0.8 * w_norm        # peso alto → gris oscuro
    lw     = 0.5 + 4.0 * w_norm
    x1, y1 = pos[u]
    x2, y2 = pos[v]
    ax.plot([x1, x2], [y1, y2], lw=lw, color=str(gray), alpha=0.9, zorder=1)

# Nodos: tamaño proporcional a n_células
max_count = max(count_dict.values())
min_size, max_size = 100, 2000

for node in G.nodes():
    day, pop = node
    x, y     = pos[node]
    n        = count_dict.get((day, pop), 1)
    size     = min_size + (max_size - min_size) * (n / max_count)
    ax.scatter(x, y, s=size, color=day_color[day],
               edgecolors="black", linewidths=0.8, zorder=2)

# Etiquetas eje Y (días)
ax.set_yticks([day_y[d] for d in days_sorted])
ax.set_yticklabels([f"d{d}" for d in days_sorted])

# Etiquetas de columnas arriba
bottom_y = min(y for _, y in pos.values()) - 1.2
for cl in cluster_order:
    ax.text(cluster_x[cl], bottom_y, cl,
            ha="center", va="bottom", fontsize=15, rotation=0)

ax.set_xticks([])
ax.set_xticklabels([])
ax.set_title("CellRank transition graph", fontsize=13)
ax.set_ylabel("Day")
for spine in ["top", "right", "bottom"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
